# FoodBridge Analytics — Dashboard Data Transformation
**Purpose:** turn the cleaned FoodBridge tables into analysis-ready tables for the Power BI dashboard.

**What this notebook does**
1. Builds the connected donation table — one row per allocation, with the donation, donor, and recipient organization details joined in. This is the dashboard's main table.
2. Builds the partner summary table — one row per partner, combining organization profile, demand fields, and allocation totals.
3. Builds the province summary table — totals per province (meals, pounds, users served, unmet demand, partner and donor counts).
4. Adds derived measures (meals per user served, share of meals vs. share of users served, allocation index, donation month).
5. Adds external context from Statistics Canada for fair comparisons between provinces.
6. Creates date and map fields (a date table, validated coordinates for map visuals).

**Note on source data:** all Second Harvest donation and organization activity values are synthetic/mock and must not be presented as verified real-world activity.


## 1. Setup and load the cleaned data

Loads `FoodBridge_Cleaned_Data.xlsx` — the final cleaned workbook agreed by the team, with one consistent `Province/Territory` column across all six tables.


In [15]:
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', 60)

CLEANED_FILE = 'FoodBridge_Cleaned_Data.xlsx'
try:
    from google.colab import files
    if not Path(CLEANED_FILE).exists():
        files.upload()
except ImportError:
    pass

sheets = pd.read_excel(CLEANED_FILE, sheet_name=None)
donations = sheets['Food Donations']
allocations = sheets['Donation Allocations']
donors = sheets['Food Donors']
orgs = sheets['SH Organizations']

EXPECTED_ROWS = {'Food Donations': 1200, 'Donation Allocations': 4805, 'Food Donors': 227, 'SH Organizations': 912}
for name, n in EXPECTED_ROWS.items():
    print(name, '-> loaded:', len(sheets[name]), '| expected:', n, '| Pass' if len(sheets[name]) == n else '| REVIEW')

Food Donations -> loaded: 1200 | expected: 1200 | Pass
Donation Allocations -> loaded: 4805 | expected: 4805 | Pass
Food Donors -> loaded: 227 | expected: 227 | Pass
SH Organizations -> loaded: 912 | expected: 912 | Pass


## 2. Connected donation table

Each allocation links one donation to one recipient organization. Joining Donation Allocations → Food Donations → Food Donors → SH Organizations gives one row per allocation (4,805 rows), with full donor and recipient detail attached. This becomes the dashboard's main fact table.

**Note on province columns:** this table ends up with four separate province-related columns, because each source table carries its own:
- `Donation Province/Territory` and `Allocation Province/Territory` — the province tied to that specific donation/allocation record.
- `Province/Territory` (no prefix) — the donor organization's own home province, from the Food Donors table.
- `Province/Territory_org` — the recipient organization's own home province, from SH Organizations.

These can legitimately differ — for example, a donor based in Ontario can have its donation allocated to a recipient organization in Alberta. A validation check below confirms `Donation Province/Territory` always matches `Allocation Province/Territory` (0 mismatches), so the only real variation left is donor province vs. recipient province, which reflects where the food actually moved — not a data error.


In [16]:
donations_r = donations.rename(columns={
    'Province/Territory': 'Donation Province/Territory',
    'Food Type': 'Donation Food Type',
})
allocations_r = allocations.rename(columns={
    'Province/Territory': 'Allocation Province/Territory',
    'Food Type': 'Allocation Food Type',
})

connected_donations = (
    allocations_r
    .merge(donations_r, on='Donation ID', how='left', validate='many_to_one')
    .merge(donors, on='Donor ID', how='left', validate='many_to_one')
    .merge(orgs, left_on='Recipient Organization ID', right_on='Organization ID',
           how='left', validate='many_to_one', suffixes=('', '_org'))
)

print('Connected donation table:', len(connected_donations), 'rows (should equal Allocations:', len(allocations_r), ')')
display(connected_donations.head())

Connected donation table: 4805 rows (should equal Allocations: 4805 )


,Donation ID,Recipient Organization ID,Allocation Province/Territory,Allocation Food Type,Allocation Date,Allocated Quantity in Pounds,Allocated Estimated Meals,Donor ID,Donation Province/Territory,Donation Food Type,Quantity in Pounds,Estimated Meals,Donation Date,Donor Type,Organization Name,Location,Longitude,Latitude,Report Category,Location Data Status,Source,Province/Territory,Organization ID,Organization name,Organization type,Food-related service or program,Official website,Operating status (simulated),Province/Territory_org,City or municipality,Street address,Postal code,Postal Code Invalid,Latitude_org,Longitude_org,Has Coordinates,Rural/urban classification,Food Types Accepted
0,D01177,ORG000020,AB,Frozen foods,2025-12-25,154317,154317,FD0110,AB,Frozen foods,634979,634979,2025-12-25,Farm / grower / primary producer,Farmlink Project,"Toronto, ON",-79.3832,43.6532,"10,000 lbs+ list",Approximate city-level location; synthetic unl...,"Second Harvest Annual Report 2025, PDF pp. 43–46",ON,ORG000020,Food Banks Alberta,Food bank,Food bank / food assistance; current programs ...,Not found / requires verification,Operating,AB,Leduc,"105, 3903 75 Avenue, Leduc, AB T9E 0K3",T9E 0K3,No,53.2594,-113.5490,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...
1,D01177,ORG009123,AB,Frozen foods,2025-12-25,175541,175541,FD0110,AB,Frozen foods,634979,634979,2025-12-25,Farm / grower / primary producer,Farmlink Project,"Toronto, ON",-79.3832,43.6532,"10,000 lbs+ list",Approximate city-level location; synthetic unl...,"Second Harvest Annual Report 2025, PDF pp. 43–46",ON,ORG009123,Bentley Colony School,Educational facility — Public,Not verified; potential community partner only,Not found / requires verification,Operating,AB,Lacombe County,28414 TWP Rd 400 Lacombe County AB T4M0S6,T4M 0S6,No,NaN,NaN,No,Unverified,Fresh produce; Prepared foods; Meat and poultr...
2,D01177,ORG006967,AB,Frozen foods,2025-12-25,143397,143397,FD0110,AB,Frozen foods,634979,634979,2025-12-25,Farm / grower / primary producer,Farmlink Project,"Toronto, ON",-79.3832,43.6532,"10,000 lbs+ list",Approximate city-level location; synthetic unl...,"Second Harvest Annual Report 2025, PDF pp. 43–46",ON,ORG006967,Association des Juristes d'expression Francais...,Registered charity (CRA category 160; subcateg...,Not verified; potential community partner only,Not found / requires verification,Operating,AB,Edmonton,314 - 8627 91 ST NW,T6C 3N1,No,53.5461,-113.4938,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...
3,D01177,ORG000022,AB,Frozen foods,2025-12-25,161724,161724,FD0110,AB,Frozen foods,634979,634979,2025-12-25,Farm / grower / primary producer,Farmlink Project,"Toronto, ON",-79.3832,43.6532,"10,000 lbs+ list",Approximate city-level location; synthetic unl...,"Second Harvest Annual Report 2025, PDF pp. 43–46",ON,ORG000022,Calgary Food Bank,Food bank,Food bank / food assistance; current programs ...,https://www.calgaryfoodbank.com/contact/,Operating,AB,Calgary,"5000 11 Street SE, Calgary, AB T2H 2Y5",T2H 2Y5,No,51.0447,-114.0719,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...
4,D00739,ORG006547,AB,Mixed food products,2025-08-09,115602,115602,FD0128,AB,Mixed food products,494791,494791,2025-08-09,Corporate / other food-sector donor,Happy Valley Harvest,"Mississauga, ON",-79.6441,43.5890,"10,000 lbs+ list",Approximate city-level location; synthetic unl...,"Second Harvest Annual Report 2025, PDF pp. 43–46",ON,ORG006547,CALGARY CHINESE CULTURAL CENTRE ASSOCIATION,Registered charity (CRA category 190; subcateg...,Not verified; potential community partner only,Not found / requires verification,Operating,AB,Calgary,197 1 ST SW,T2P 4M4,No,51.0447,-114.0719,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...


In [17]:
mismatch = (connected_donations['Donation Province/Territory'] != connected_donations['Allocation Province/Territory']).sum()
print('Donation vs Allocation province mismatches:', mismatch, '(should be 0)')

Donation vs Allocation province mismatches: 0 (should be 0)


**Validation result:** 0 mismatches between donation and allocation province — confirms the join did not introduce inconsistent province values. Step 2 (connected donation table) is complete.

## 3. Partner summary table

One row per partner (912 rows), combining three things:
- **Organization profile** — from SH Organizations.
- **Demand fields** — from Population Demand (users served, unmet demand).
- **Allocation totals** — computed here from Donation Allocations: total meals received, total pounds received, number of donations, and distinct food types received per partner.

`Province/Territory` and `Postal Code Invalid` are exact-name duplicates of columns already in SH Organizations, so they're dropped from the Population Demand side before joining. `Postal Code` is also dropped — SH Organizations has the same field under a different capitalization (`Postal code`), so keeping both would leave two near-identical columns in the table.



In [18]:
pop = sheets['Population Demand']

partner_summary = orgs.merge(
    pop.drop(columns=['Province/Territory', 'Postal Code', 'Postal Code Invalid'], errors='ignore'),
    on='Organization ID', how='inner', validate='one_to_one'
)
assert len(partner_summary) == 912 and partner_summary['Organization ID'].is_unique

allocation_totals = (
    allocations.groupby('Recipient Organization ID')
    .agg(
        total_meals_received=('Allocated Estimated Meals', 'sum'),
        total_pounds_received=('Allocated Quantity in Pounds', 'sum'),
        number_of_donations=('Donation ID', 'nunique'),
        distinct_food_types_received=('Food Type', 'nunique'),
    )
    .rename_axis('Organization ID').reset_index()
)

partner_summary = partner_summary.merge(allocation_totals, on='Organization ID', how='left', validate='one_to_one')
totals_cols = ['total_meals_received', 'total_pounds_received', 'number_of_donations', 'distinct_food_types_received']
partner_summary[totals_cols] = partner_summary[totals_cols].fillna(0)
partner_summary[['number_of_donations', 'distinct_food_types_received']] = partner_summary[['number_of_donations', 'distinct_food_types_received']].astype(int)

print('Partner summary table:', partner_summary.shape, '(should be 912 rows)')
display(partner_summary.head())

Partner summary table: (912, 30) (should be 912 rows)


,Organization ID,Organization name,Organization type,Food-related service or program,Official website,Operating status (simulated),Province/Territory,City or municipality,Street address,Postal code,Postal Code Invalid,Latitude,Longitude,Has Coordinates,Rural/urban classification,Food Types Accepted,Users Served in 2025,Users Served Missing,Users Served Source,Meals Provided by Second Harvest (Synthetic Estimate),Target Populations,Food Demand Status,Unmet Demand (Estimated People; Synthetic),Unmet Demand Status,Estimation Method,Unmet Demand Estimation Rate (Synthetic),total_meals_received,total_pounds_received,number_of_donations,distinct_food_types_received
0,ORG000986,Aurora College,College / public institution,Education; potential community food programming,https://www.auroracollege.nt.ca/,Operating,NT,Yellowknife,"5004 54 Street, Yellowknife, NT X1A 2R3",X1A 2R3,No,62.4540,-114.3718,Yes,Urban,Fresh produce; Prepared foods; Pantry staples;...,80.0,No,Original estimate,413,Children/youth or students (inferred); People ...,Estimated unmet demand; synthetic proxy (not r...,6.0,Estimated,Meals: synthetic allocation of Second Harvest'...,0.0785,413,413,1,1
1,ORG000987,Aurora College – Thebacha Campus,College / public institution,Education; potential student/community programs,https://www.auroracollege.nt.ca/,Operating,NT,Fort Smith,"50 Conibear Crescent, Fort Smith, NT X0E 0P0",X0E 0P0,No,60.0044,-111.8886,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...,80.0,No,Original estimate,386,Children/youth or students (inferred),Estimated unmet demand; synthetic proxy (not r...,6.0,Estimated,Meals: synthetic allocation of Second Harvest'...,0.0744,386,386,2,2
2,ORG001012,Food Rescue Yellowknife,Food recovery / redistribution,Food recovery and redistribution to community ...,https://foodrescueyellowknife.ca/contact/,Operating,NT,Yellowknife,"P.O. Box 1383, Yellowknife, NT X1A 2P1 (mailin...",X1A 2P1,No,62.4540,-114.3718,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...,80.0,No,Original estimate,477,People and households needing food assistance ...,Estimated unmet demand; synthetic proxy (not r...,21.0,Estimated,Meals: synthetic allocation of Second Harvest'...,0.2604,477,477,2,2
3,ORG000026,Yellowknife Food Bank,Food bank,Food bank / food assistance; current programs ...,Not found / requires verification,Operating,NT,Yellowknife,"4922 50 Street, Yellowknife, NT X1A 2P9",X1A 2P9,No,62.4540,-114.3718,Yes,Urban,Fresh produce; Prepared foods; Meat and poultr...,16319.0,No,Original estimate,72396,People and households needing food assistance ...,Estimated unmet demand; synthetic proxy (not r...,2051.0,Estimated,Meals: synthetic allocation of Second Harvest'...,0.1257,72396,72396,3,3
4,ORG001372,INUVIK FOOD BANK,Food bank,Food bank / food assistance; current programs ...,Not found / requires verification,Operating,NT,Inuvik,P.O. BOX 3288,X0E 0T0,No,NaN,NaN,No,Urban,Fresh produce; Prepared foods; Meat and poultr...,48.0,No,Randomized,240,People and households needing food assistance ...,Estimated unmet demand; synthetic proxy (not r...,4.0,Estimated,Meals: synthetic allocation of Second Harvest'...,0.0835,240,240,1,1


In [19]:
meal_diff = (partner_summary['total_meals_received'] - partner_summary['Meals Provided by Second Harvest (Synthetic Estimate)']).abs()
print('Partners whose allocated meals differ from the demand file:', int((meal_diff > 0).sum()), '(should be 0)')

Partners whose allocated meals differ from the demand file: 0 (should be 0)


**Validation result:** 0 partners where allocated meals differ from the demand file's own meals figure — confirms the allocation totals reconcile with the source data. Step 3 (partner summary table) is complete.


## 4. Province summary table

Totals per province, built from the partner summary table and the Food Donors table:
- Meals and pounds received (summed from partner_summary).
- Users served and known unmet demand (summed from partner_summary; unmet demand only counts partners where it's actually known, not missing).
- Number of partners per province.
- Number of distinct donors per province (from Food Donors, using the donor's own home province).

In [20]:
province_summary = (
    partner_summary.groupby('Province/Territory')
    .agg(
        total_meals_received=('total_meals_received', 'sum'),
        total_pounds_received=('total_pounds_received', 'sum'),
        total_users_served=('Users Served in 2025', 'sum'),
        total_known_unmet_demand=('Unmet Demand (Estimated People; Synthetic)', 'sum'),
        number_of_partners=('Organization ID', 'nunique'),
    )
    .reset_index()
)

donor_counts = donors.groupby('Province/Territory')['Donor ID'].nunique().rename('number_of_donors').reset_index()
province_summary = province_summary.merge(donor_counts, on='Province/Territory', how='left')
province_summary['number_of_donors'] = province_summary['number_of_donors'].fillna(0).astype(int)

print('Province summary table:', province_summary.shape, '(should be 13 rows — 10 provinces + 3 territories)')
display(province_summary)

Province summary table: (13, 7) (should be 13 rows — 10 provinces + 3 territories)


,Province/Territory,total_meals_received,total_pounds_received,total_users_served,total_known_unmet_demand,number_of_partners,number_of_donors
0,AB,14090000,14090000,117902.0,24049.0,92,19
1,BC,10940000,10940000,81965.0,12585.0,105,12
2,MB,6890000,6890000,80361.0,19214.0,72,11
3,NB,3050000,3050000,52563.0,7201.0,38,1
4,NL,645000,645000,75780.0,11272.0,55,1
5,NS,2600000,2600000,29335.0,2818.0,68,2
6,NT,78400,78400,17411.0,2263.0,18,0
7,NU,163100,163100,20359.0,3846.0,23,0
8,ON,45490000,45490000,140477.0,21400.0,296,140
9,PE,148900,148900,7872.0,1900.0,19,6


**Validation result:** province summary has 13 rows — one per province/territory, matching the 10 provinces + 3 territories in the partner summary with none dropped or duplicated. Step 4 (province summary table) is complete.


## 5. Derived measures

Four new fields for the dashboard:
- **Meals per user served** — total meals received ÷ users served. How much food reaches each person this partner serves.
- **Share of meals** and **share of users served** — each partner's percentage of the national total meals received, and separately, their percentage of the national total users served.
- **Allocation index** — share of meals ÷ share of users served. 1.0 means a partner receives exactly its proportional share of meals relative to the share of need it represents; above 1.0 means more than proportional, below means less.
- **Donation month** — the donation date's calendar month, added to the connected donation table for time-based filtering on the dashboard.

Note: 1 partner has no "Users Served" figure (missing in the source data), so its per-user ratios come out as missing (NaN) rather than a guessed number.

In [21]:
partner_summary['meals_per_user_served'] = (
    partner_summary['total_meals_received'] / partner_summary['Users Served in 2025']
).round(2)

partner_summary['share_of_meals'] = (
    partner_summary['total_meals_received'] / partner_summary['total_meals_received'].sum()
)
partner_summary['share_of_users_served'] = (
    partner_summary['Users Served in 2025'] / partner_summary['Users Served in 2025'].sum()
)
partner_summary['allocation_index'] = (
    partner_summary['share_of_meals'] / partner_summary['share_of_users_served']
).round(2)

connected_donations['donation_month'] = pd.to_datetime(connected_donations['Donation Date']).dt.to_period('M').astype(str)

print('Partners missing meals-per-user (no Users Served figure):', partner_summary['meals_per_user_served'].isna().sum(), '(should be 1)')
print('Allocation index — min/median/max:', partner_summary['allocation_index'].min(), partner_summary['allocation_index'].median(), partner_summary['allocation_index'].max())
display(partner_summary[['Organization ID', 'meals_per_user_served', 'share_of_meals', 'share_of_users_served', 'allocation_index']].head())
display(connected_donations[['Donation ID', 'Donation Date', 'donation_month']].head())

Partners missing meals-per-user (no Users Served figure): 1 (should be 1)
Allocation index — min/median/max: 0.03 1.0 3.45


,Organization ID,meals_per_user_served,share_of_meals,share_of_users_served,allocation_index
0,ORG000986,5.16,0.000004,0.000107,0.04
1,ORG000987,4.82,0.000004,0.000107,0.04
2,ORG001012,5.96,0.000005,0.000107,0.05
3,ORG000026,4.44,0.000750,0.021803,0.03
4,ORG001372,5.00,0.000002,0.000064,0.04


,Donation ID,Donation Date,donation_month
0,D01177,2025-12-25,2025-12
1,D01177,2025-12-25,2025-12
2,D01177,2025-12-25,2025-12
3,D01177,2025-12-25,2025-12
4,D00739,2025-08-09,2025-08


**Validation result:** missing meals-per-user count is 1 (matches the 1 partner with no Users Served figure), and the allocation index centers on 1.0 as expected. Step 5 (derived measures) is complete.

## 6. External context (Statistics Canada)

Adds real provincial context to `province_summary`, for fair comparisons between provinces of different size and need:

- **Food insecurity, 2024** (% of persons): Statistics Canada, Canadian Income Survey, [Table 13-10-0835-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1310083501). `fi_data_quality` is Statistics Canada's own quality letter (A = excellent … E = use with caution); territory estimates are less reliable.
- **Population, July 1 2025**: Statistics Canada, [Table 17-10-0009-01](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710000901).




In [22]:
PROVINCIAL_CONTEXT = pd.DataFrame([
    ('NL', 25.5, 20.4,  7.5, 'C',    550202), ('PE', 23.3, 18.9,  5.8, 'D',    181594),
    ('NS', 26.2, 19.7,  7.5, 'C',   1095157), ('NB', 28.2, 20.9,  6.8, 'C',    874711),
    ('QC', 18.0, 12.1,  4.1, 'C',   9051022), ('ON', 25.4, 20.2,  7.1, 'B',  16216961),
    ('MB', 27.9, 22.3,  7.0, 'C',   1508582), ('SK', 26.7, 19.7,  6.6, 'C',   1266289),
    ('AB', 28.4, 22.8,  9.4, 'C',   5024631), ('BC', 23.2, 17.4,  5.8, 'C',   5702544),
    ('YT', 15.5, 13.0,  5.6, 'D',     49150), ('NT', 16.4, 12.6,  5.5, 'E',     46031),
    ('NU', 56.4, 53.2, 31.0, 'C',     42108),
], columns=['Province/Territory', 'food_insecure_pct_2024', 'mod_severe_food_insecure_pct_2024',
            'severe_food_insecure_pct_2024', 'fi_data_quality', 'population_2025'])

province_summary = province_summary.merge(PROVINCIAL_CONTEXT, on='Province/Territory', how='left', validate='one_to_one')

missing_context = province_summary['food_insecure_pct_2024'].isna().sum()
print('Provinces/territories missing external context:', missing_context, '(should be 0)')
display(province_summary)

Provinces/territories missing external context: 0 (should be 0)


,Province/Territory,total_meals_received,total_pounds_received,total_users_served,total_known_unmet_demand,number_of_partners,number_of_donors,food_insecure_pct_2024,mod_severe_food_insecure_pct_2024,severe_food_insecure_pct_2024,fi_data_quality,population_2025
0,AB,14090000,14090000,117902.0,24049.0,92,19,28.4,22.8,9.4,C,5024631
1,BC,10940000,10940000,81965.0,12585.0,105,12,23.2,17.4,5.8,C,5702544
2,MB,6890000,6890000,80361.0,19214.0,72,11,27.9,22.3,7.0,C,1508582
3,NB,3050000,3050000,52563.0,7201.0,38,1,28.2,20.9,6.8,C,874711
4,NL,645000,645000,75780.0,11272.0,55,1,25.5,20.4,7.5,C,550202
5,NS,2600000,2600000,29335.0,2818.0,68,2,26.2,19.7,7.5,C,1095157
6,NT,78400,78400,17411.0,2263.0,18,0,16.4,12.6,5.5,E,46031
7,NU,163100,163100,20359.0,3846.0,23,0,56.4,53.2,31.0,C,42108
8,ON,45490000,45490000,140477.0,21400.0,296,140,25.4,20.2,7.1,B,16216961
9,PE,148900,148900,7872.0,1900.0,19,6,23.3,18.9,5.8,D,181594


**Validation result:** 0 of 13 provinces/territories missing external context — population and food-insecurity figures merged successfully for every province. Step 6 (external context) is complete.

## 7. Date and map fields

**Date table** — one row per calendar day spanning the earliest to latest donation/allocation date, with Year, Month, Month Name, Quarter, and combined Year-Month / Year-Quarter labels. This is a standard Power BI date dimension, used to slice the dashboard by month or quarter.

**Validated coordinates** — a partner's latitude/longitude are only usable for the map visual if they fall inside Canada's real geographic bounds (roughly 41°–84° N, 142°–52° W) and `Has Coordinates` is "Yes". A new flag, `coordinates_valid_for_map`, marks which partners are safe to plot; anything missing or out of range is excluded rather than guessed.


In [23]:
all_dates = pd.concat([
    pd.to_datetime(connected_donations['Donation Date']),
    pd.to_datetime(connected_donations['Allocation Date']),
]).dropna()

date_table = pd.DataFrame({'Date': pd.date_range(all_dates.min(), all_dates.max(), freq='D')})
date_table['Year'] = date_table['Date'].dt.year
date_table['Month'] = date_table['Date'].dt.month
date_table['Month Name'] = date_table['Date'].dt.strftime('%B')
date_table['Quarter'] = date_table['Date'].dt.quarter
date_table['Year-Month'] = date_table['Date'].dt.to_period('M').astype(str)
date_table['Year-Quarter'] = date_table['Year'].astype(str) + '-Q' + date_table['Quarter'].astype(str)

print('Date table:', date_table.shape, '| range:', date_table['Date'].min().date(), 'to', date_table['Date'].max().date())
display(date_table.head())

CANADA_LAT_RANGE = (41, 84)
CANADA_LON_RANGE = (-142, -52)

partner_summary['coordinates_valid_for_map'] = (
    partner_summary['Latitude'].between(*CANADA_LAT_RANGE) &
    partner_summary['Longitude'].between(*CANADA_LON_RANGE) &
    partner_summary['Has Coordinates'].eq('Yes')
)

out_of_range = partner_summary[
    partner_summary['Latitude'].notna() & ~partner_summary['Latitude'].between(*CANADA_LAT_RANGE)
]
print('Partners with coordinates marked present:', int(partner_summary['Has Coordinates'].eq('Yes').sum()), 'of 912')
print('Partners validated as mappable:', int(partner_summary['coordinates_valid_for_map'].sum()))
print('Out-of-range coordinates found:', len(out_of_range), '(should be 0)')
display(partner_summary[['Organization ID', 'Latitude', 'Longitude', 'Has Coordinates', 'coordinates_valid_for_map']].head())

Date table: (365, 7) | range: 2025-01-01 to 2025-12-31


,Date,Year,Month,Month Name,Quarter,Year-Month,Year-Quarter
0,2025-01-01,2025,1,January,1,2025-01,2025-Q1
1,2025-01-02,2025,1,January,1,2025-01,2025-Q1
2,2025-01-03,2025,1,January,1,2025-01,2025-Q1
3,2025-01-04,2025,1,January,1,2025-01,2025-Q1
4,2025-01-05,2025,1,January,1,2025-01,2025-Q1


Partners with coordinates marked present: 732 of 912
Partners validated as mappable: 732
Out-of-range coordinates found: 0 (should be 0)


,Organization ID,Latitude,Longitude,Has Coordinates,coordinates_valid_for_map
0,ORG000986,62.4540,-114.3718,Yes,True
1,ORG000987,60.0044,-111.8886,Yes,True
2,ORG001012,62.4540,-114.3718,Yes,True
3,ORG000026,62.4540,-114.3718,Yes,True
4,ORG001372,NaN,NaN,No,False


**Validation result:** 732 of 912 partners have coordinates, and all 732 fall within Canada's geographic bounds — 0 out-of-range values found. The date table covers the full donation/allocation date range (2025-01-01 to 2025-12-31) with no gaps. Step 7 (date and map fields) is complete.

## 8. Benchmarking Second Harvest's footprint against real need

**Purpose:** Steps 1–7 built SH's own activity (meals, partners, allocations) at the
province level and attached external context (population, food-insecurity rate)
from Statistics Canada. But we never actually *used* that external data to compare
anything — we just stored it next to our numbers.

**What this step does:** turns raw totals into fair, comparable metrics by scaling
them against population and estimated need, so a big province like Ontario isn't
automatically "doing better" just because it's bigger. Specifically:
- `meals_per_capita` — meals delivered per person in the province
- `partners_per_100k_population` — how many partner orgs SH has per 100k people
- `need_based_allocation_index` — compares SH's share of national meals to that
  province's share of national *estimated food-insecure population* (StatCan rate ×
  population). 1.0 = SH's meal distribution matches that province's share of need;
  above 1.0 = overserved relative to need; below 1.0 = underserved.




In [24]:
# 8. Benchmarking SH's footprint against real need (normalization / fairness metrics)

# national totals = denominators for "share of ___"
national_meals = province_summary['total_meals_received'].sum()
national_food_insecure_pop = (
    province_summary['population_2025'] * province_summary['food_insecure_pct_2024'] / 100
).sum()

# per-capita metrics (raw totals scaled by population)
province_summary['meals_per_capita'] = (
    province_summary['total_meals_received'] / province_summary['population_2025']
)
province_summary['partners_per_100k_population'] = (
    province_summary['number_of_partners'] / province_summary['population_2025'] * 100_000
)

# estimated number of food-insecure people per province (StatCan rate x population)
province_summary['est_food_insecure_population'] = (
    province_summary['population_2025'] * province_summary['food_insecure_pct_2024'] / 100
)

# share of SH's national meal volume vs that province's share of national need
province_summary['share_of_national_meals'] = province_summary['total_meals_received'] / national_meals
province_summary['share_of_national_food_insecure_pop'] = (
    province_summary['est_food_insecure_population'] / national_food_insecure_pop
)

# need-based allocation index: >1 = SH sends more than the province's share of need; <1 = less
province_summary['need_based_allocation_index'] = (
    province_summary['share_of_national_meals'] / province_summary['share_of_national_food_insecure_pop']
)

# validation
print(province_summary[['Province/Territory', 'meals_per_capita',
                         'partners_per_100k_population',
                         'need_based_allocation_index']].sort_values('need_based_allocation_index'))

assert province_summary['need_based_allocation_index'].notna().all(), \
    "Some provinces are missing the index — check for zero/NaN population or food-insecurity data"

   Province/Territory  meals_per_capita  partners_per_100k_population  \
9                  PE          0.819961                     10.462901   
4                  NL          1.172297                      9.996329   
10                 QC          0.907080                      0.939120   
7                  NU          3.873373                     54.621450   
1                  BC          1.918442                      1.841283   
12                 YT          1.340793                      6.103764   
5                  NS          2.374089                      6.209155   
0                  AB          2.804186                      1.830980   
6                  NT          1.703200                     39.104082   
8                  ON          2.805088                      1.825250   
3                  NB          3.486866                      4.344292   
11                 SK          3.324675                      3.000895   
2                  MB          4.567203            

**Validation result:** `need_based_allocation_index` computed for all 13 provinces/
territories with no missing values — index ranges from 0.36 (PE) to 1.70 (MB).
Step 8 (benchmarking against real need) is complete.




## 9. Export final tables for the dashboard

The four tables built above are the final outputs of this notebook — everything
the dashboard needs:

- `connected_donations` — transaction-level table (one row per allocation split)
- `partner_summary` — one row per partner organization
- `province_summary` — one row per province/territory, including the StatCan
  context and the need-based comparison metrics from Step 8
- `date_table` — date/month/quarter reference table for map and time visuals

Exporting each to CSV so they can be loaded directly into the dashboard tool.

In [25]:
# Export the 4 final tables for the dashboard

connected_donations.to_csv('connected_donations.csv', index=False)
partner_summary.to_csv('partner_summary.csv', index=False)
province_summary.to_csv('province_summary.csv', index=False)
date_table.to_csv('date_table.csv', index=False)

print("Exported:")
print(f"  connected_donations.csv — {connected_donations.shape[0]} rows, {connected_donations.shape[1]} cols")
print(f"  partner_summary.csv     — {partner_summary.shape[0]} rows, {partner_summary.shape[1]} cols")
print(f"  province_summary.csv    — {province_summary.shape[0]} rows, {province_summary.shape[1]} cols")
print(f"  date_table.csv          — {date_table.shape[0]} rows, {date_table.shape[1]} cols")

# Download each one straight to your computer (Colab-specific)
from google.colab import files
files.download('connected_donations.csv')
files.download('partner_summary.csv')
files.download('province_summary.csv')
files.download('date_table.csv')

Exported:
  connected_donations.csv — 4805 rows, 39 cols
  partner_summary.csv     — 912 rows, 35 cols
  province_summary.csv    — 13 rows, 18 cols
  date_table.csv          — 365 rows, 7 cols


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>